##  Install & Import Required Libraries

In [23]:
#!pip install nltk gensim beautifulsoup4 tensorflow pandas scikit-learn
import os
import csv
import numpy as np
import pandas as pd
import re
import pickle
from bs4 import BeautifulSoup
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from gensim.models import Word2Vec
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Bidirectional, GlobalMaxPool1D
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split


## Download NLTK Resources

In [2]:

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Revanasidda\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Revanasidda\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Revanasidda\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## Set Constants and File Paths

In [7]:

EMBED_DIM   = 100
MAX_SEQ_LEN = 100
TSV_PATH    = 'drugsComTrain_raw.tsv'
MODEL_DIR   = 'model'
os.makedirs("./", exist_ok=True)


##  Define Text Cleaning Function

In [26]:

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = BeautifulSoup(text, 'html.parser').get_text()
    text = re.sub('[^a-zA-Z]', ' ', text)
    tokens = text.lower().split()
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return ' '.join(tokens)


## Load Dataset & Filter Conditions

In [10]:

df = pd.read_csv("./drugsComTrain_raw.tsv", sep='\t', engine='python', quoting=csv.QUOTE_NONE, dtype=str)
df = df.dropna(subset=['review'])

# Filter to top 25+ conditions with >100 reviews
condition_counts = df['condition'].value_counts()
keep = condition_counts[condition_counts > 100].index.tolist()
df = df[df['condition'].isin(keep)].copy()

label_map = {c: i for i, c in enumerate(keep)}
df['label'] = df['condition'].map(label_map)


## Clean Review Text

In [12]:

df['cleaned'] = df['review'].apply(clean_text)


C:\Users\Revanasidda\AppData\Local\Temp\ipykernel_5580\780511836.py:7: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  text = BeautifulSoup(text, 'html.parser').get_text()


##  Tokenize and Pad Sequences

In [15]:

tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['cleaned'])
sequences = tokenizer.texts_to_sequences(df['cleaned'])
X = pad_sequences(sequences, maxlen=MAX_SEQ_LEN)
y = df['label'].astype(int).values

# Save tokenizer
with open(os.path.join(MODEL_DIR, 'tokenizer.pkl'), 'wb') as f:
    pickle.dump(tokenizer, f)


##  Train Word2Vec and Create Embedding Matrix

In [16]:

token_lists = [t.split() for t in df['cleaned']]
w2v = Word2Vec(sentences=token_lists, vector_size=EMBED_DIM, window=5, min_count=1, workers=4)

word_index = tokenizer.word_index
num_words = len(word_index) + 1
embedding_matrix = np.zeros((num_words, EMBED_DIM))
for word, idx in word_index.items():
    if word in w2v.wv:
        embedding_matrix[idx] = w2v.wv[word]


##  Train-Test Split

In [17]:

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


## Build BiLSTM Model

In [18]:

num_classes = len(keep)

inp = Input(shape=(MAX_SEQ_LEN,))
emb = Embedding(
    input_dim=num_words,
    output_dim=EMBED_DIM,
    weights=[embedding_matrix],
    input_length=MAX_SEQ_LEN,
    trainable=False
)(inp)
x = Bidirectional(LSTM(64, return_sequences=True))(emb)
x = GlobalMaxPool1D()(x)
out = Dense(num_classes, activation='softmax')(x)
model = Model(inputs=inp, outputs=out)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


C:\Users\Revanasidda\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 100)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ embedding (Embedding)                │ (None, 100, 100)            │       3,714,800 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional (Bidirectional)        │ (None, 100, 128)            │          84,480 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_max_pooling1d                 │ (None, 128)                 │               0 │
│ (GlobalMaxPooling1D)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 150)                 │          19,350 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,818,630 (14.57 MB)

 Trainable params: 103,830 (405.59 KB)

 Non-trainable params: 3,714,800 (14.17 MB)

## Train the Model

In [19]:

es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=128,
    callbacks=[es]
)


Epoch 1/10
928/928 ━━━━━━━━━━━━━━━━━━━━ 90s 94ms/step - accuracy: 0.4000 - loss: 2.8632 - val_accuracy: 0.5844 - val_loss: 1.7164
Epoch 2/10
928/928 ━━━━━━━━━━━━━━━━━━━━ 88s 95ms/step - accuracy: 0.6039 - loss: 1.6073 - val_accuracy: 0.6262 - val_loss: 1.4671
Epoch 3/10
928/928 ━━━━━━━━━━━━━━━━━━━━ 89s 95ms/step - accuracy: 0.6396 - loss: 1.4026 - val_accuracy: 0.6493 - val_loss: 1.3489
Epoch 4/10
928/928 ━━━━━━━━━━━━━━━━━━━━ 86s 93ms/step - accuracy: 0.6668 - loss: 1.2740 - val_accuracy: 0.6626 - val_loss: 1.2799
Epoch 5/10
928/928 ━━━━━━━━━━━━━━━━━━━━ 90s 97ms/step - accuracy: 0.6823 - loss: 1.1942 - val_accuracy: 0.6687 - val_loss: 1.2381
Epoch 6/10
928/928 ━━━━━━━━━━━━━━━━━━━━ 91s 98ms/step - accuracy: 0.6938 - loss: 1.1426 - val_accuracy: 0.6754 - val_loss: 1.2077
Epoch 7/10
928/928 ━━━━━━━━━━━━━━━━━━━━ 96s 103ms/step - accuracy: 0.7009 - loss: 1.0971 - val_accuracy: 0.6783 - val_loss: 1.1867
Epoch 8/10
928/928 ━━━━━━━━━━━━━━━━━━━━ 92s 100ms/step - accuracy: 0.7149 - loss: 1.0418 

## Save Model, Tokenizer and Label Map

In [20]:

model.save(os.path.join(MODEL_DIR, 'model.h5'))
with open(os.path.join(MODEL_DIR, 'label_map.pkl'), 'wb') as f:
    pickle.dump(label_map, f)
print("✅ Model, tokenizer, and label map saved in 'model/'")


✅ Model, tokenizer, and label map saved in 'model/'


In [25]:
# 📦 Import required libraries
import numpy as np
import pandas as pd
import pickle
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 📁 Load the trained model and tokenizer
model = load_model('model/model.h5')
with open('model/tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

# 🧠 Try loading saved label map (label -> index), else recreate from dataset
try:
    with open('model/label_map.pkl', 'rb') as f:
        label_map = pickle.load(f)
    print("✅ Loaded label_map from pickle file.")
except FileNotFoundError:
    print("⚠️ label_map.pkl not found. Rebuilding from training data...")
    df = pd.read_csv('drugsComTrain_raw.tsv', sep='\t', engine='python', quoting=3, dtype=str)
    df = df[df['condition'].notnull()]
    top_conditions = df['condition'].value_counts().head(25).index.tolist()  # Top 25 conditions
    label_map = {c: i for i, c in enumerate(top_conditions)}

# 🔄 Reverse label_map to get index -> condition label
index_to_label = {v: k for k, v in label_map.items()}

# 🔍 Define a function to predict the most likely medical condition from a review
def predict_condition(review_text):
    # Tokenize and pad input
    sequence = tokenizer.texts_to_sequences([review_text])
    padded = pad_sequences(sequence, maxlen=100)

    # Get prediction probabilities from model
    preds = model.predict(padded)
    pred_index = np.argmax(preds[0])
    confidence = preds[0][pred_index]

    # Map index to label
    predicted_label = index_to_label.get(pred_index, "Unknown")
    return predicted_label, confidence

# 🧪 Define a function to print top-k predictions with confidence
def predict_top_k_conditions(review_text, k=3):
    sequence = tokenizer.texts_to_sequences([review_text])
    padded = pad_sequences(sequence, maxlen=100)
    preds = model.predict(padded)[0]

    # Get indices of top-k predictions
    top_indices = preds.argsort()[-k:][::-1]
    print(f"Top {k} Predicted Conditions:")
    for idx in top_indices:
        label = index_to_label.get(idx, "Unknown")
        confidence = preds[idx]
        print(f"  - {label}: {confidence:.2f}")

# 📝 Example usage
review = "This medicine worked great for my anxiety. I feel much more relaxed now."
label, conf = predict_condition(review)
print(f"\nTop Prediction:\n  - Condition: {label}\n  - Confidence: {conf:.2f}")

# 🔝 Print top 3 predictions
predict_top_k_conditions(review, k=3)


✅ Loaded label_map from pickle file.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 257ms/step

Top Prediction:
  - Condition: Anxiety
  - Confidence: 0.91
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Top 3 Predicted Conditions:
  - Anxiety: 0.91
  - Generalized Anxiety Disorde: 0.02
  - Anxiety and Stress: 0.02


In [27]:
!git init

Initialized empty Git repository in C:/Users/Revanasidda/Patient-Condition-Classification-Review/.git/


In [ ]:
!git add .
!git commit -m "Add project to Github Repo"